Monday, hands on: Anand's suite, solved

> "Nothing a person can mistype."

Six queries, each with one comment line stating its question and its denominator. Released at close of session. Every
comment line states its question and its denominator, which is the half an auditor reads first.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
kit.flow(["connect", "filter", "group", "name the steps", "ship the suite"], lit=[0],
         title="Where you are")

## 1. The handshake

Before anything else: how big is the book, and does it match what the platform lead said?

In [2]:
n = kit.sql("""SELECT count(*) AS orders FROM orders""", conn=conn)[0]
kit.check("the warehouse holds a thousand orders", list(n.values())[0] == 1000, str(n))

## 2. The largest orders

Last week's first finding was the order that pulled the mean away from the median. One statement
now. Give it an order, or the five rows you get back are any five rows.

In [3]:
top = kit.sql("""SELECT order_id, customer_id, channel, amount FROM orders ORDER BY amount DESC LIMIT 5""", conn=conn)
kit.check("five rows came back", len(top) == 5, f"{len(top)} rows")
kit.check("they are sorted from largest down",
          all(top[i]["amount"] >= top[i + 1]["amount"] for i in range(len(top) - 1)))

## 3. Per quarter

Revenue and order count for each quarter, one row each.

In [4]:
q = kit.sql("""SELECT quarter, count(*) AS orders, sum(amount) AS revenue FROM orders GROUP BY quarter ORDER BY quarter""", conn=conn)
kit.check("one row per quarter", len(q) == 2, f"{len(q)} rows")
kit.table(list(q[0]) if q else ["result"], [list(r.values()) for r in q],
          caption="The two quarters")

quarter,orders,revenue
Q1,538,100000000.00
Q2,462,98400000.00


## 4. Meet the error on purpose

Ask for segment and channel while grouping by segment alone. Read the message, then fix it two
ways and notice they answer two different questions.

In [5]:
try:
    kit.sql("""SELECT c.segment, o.channel, sum(o.amount) FROM orders o JOIN customers c USING (customer_id) GROUP BY c.segment""", conn=conn)
    print("no error: check that you grouped by segment alone")
except Exception as e:
    conn.rollback()
    print(str(e).strip().splitlines()[0])

column "o.channel" must appear in the GROUP BY clause or be used in an aggregate function


## 5. Frequency, the branch that moved

Orders per customer, per segment, per quarter. The denominator is customers who ordered in that
quarter. Write that down in the comment before you write the SQL.

In [6]:
freq = kit.sql("""SELECT c.segment, o.quarter, count(*) AS orders,
              count(DISTINCT o.customer_id) AS customers,
              round(count(*)::numeric / count(DISTINCT o.customer_id), 2) AS orders_per_customer
       FROM orders o JOIN customers c USING (customer_id)
       GROUP BY c.segment, o.quarter ORDER BY c.segment, o.quarter""", conn=conn)
kit.check("eight rows, four segments across two quarters", len(freq) == 8, f"{len(freq)} rows")
kit.ladder(["orders", "per segment", "per quarter", "divided by customers"],
           lit=[3], title="Building the frequency number")

## 6. The comparison, as named steps

Two CTEs, joined on segment, one row out per segment carrying the change.

In [7]:
change = kit.sql("""WITH q1 AS (SELECT c.segment, count(*) orders FROM orders o
                   JOIN customers c USING (customer_id)
                   WHERE o.quarter='Q1' GROUP BY c.segment),
            q2 AS (SELECT c.segment, count(*) orders FROM orders o
                   JOIN customers c USING (customer_id)
                   WHERE o.quarter='Q2' GROUP BY c.segment)
       SELECT q1.segment, q1.orders AS q1_orders, q2.orders AS q2_orders,
              round(100.0*(q2.orders-q1.orders)/q1.orders, 1) AS order_change_pct
       FROM q1 JOIN q2 USING (segment) ORDER BY order_change_pct""", conn=conn)
kit.check("one row per segment", len(change) == 4, f"{len(change)} rows")
worst = min(change, key=lambda r: float(list(r.values())[-1]))
kit.check("the steepest fall is Retail-Plus", worst["segment"] == "Retail-Plus", str(worst))
kit.tree({"label": "revenue moved", "branches": [
    ("customers?", {"label": "flat"}),
    ("frequency?", {"label": "fell", "branches": [
        ("which segment?", {"label": "Retail-Plus"})]}),
    ("basket?", {"label": "flat"})]},
    taken=["frequency?", "which segment?"],
    title="Which branch the suite points at")

## 7. Shipped

The six queries live in `exercises/solutions/C2_W02_D01_monday_suite_solution_STUDENT.sql` with
their comment lines. The comment states the question and the denominator, which is the half an
auditor reads first and the half that is missing from most work.

In [8]:
kit.decision_ladder(["a number in a message", "a spreadsheet you email",
                     "a query anybody can run", "a scheduled job"], cut_at=2,
                    title="What Anand asked for")
kit.check_summary()